# Phase 0 spike, goal 3 — B1 seasonal naive + a first LightGBM

Baselines and a first model for NO2, calendar + price-lag features only, per `docs/ROADMAP.md` Phase 0 and `docs/FORECASTING.md` §2/§4.

**Not the real backtest.** `docs/FORECASTING.md` §6 specifies a walk-forward, issue-time-anchored, 24h-embargoed protocol over 24 monthly folds. This notebook uses a single last-8-weeks holdout instead — good enough to sanity-check that a gradient booster beats seasonal naive, not good enough to trust as a reported model metric. That harness is Phase 3 work.

**Leakage discipline kept even here.** For a forecast issued `lead` hours before its target hour, every lag feature is computed relative to the *issue time*, not the target hour — otherwise a 48h-lead forecast could see a 24h-ago price that wouldn't exist yet in production. This is the point-in-time rule from `docs/FORECASTING.md` §5, applied in miniature.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb

df = pd.read_csv("out/no2_prices_12mo.csv")
df["hour_start"] = pd.to_datetime(df["hour_start"], utc=True)
df = df.sort_values("hour_start").drop_duplicates("hour_start").set_index("hour_start")

price = df["price_ex_vat_ore_per_kwh"]
print(f"{len(price)} hourly rows, {price.index.min()} .. {price.index.max()}")
price.head()

8760 hourly rows, 2025-09-14 22:00:00+00:00 .. 2026-09-14 21:00:00+00:00


hour_start
2025-09-14 22:00:00+00:00    18.141
2025-09-14 23:00:00+00:00     9.187
2025-09-15 00:00:00+00:00     5.201
2025-09-15 01:00:00+00:00     3.765
2025-09-15 02:00:00+00:00     3.707
Name: price_ex_vat_ore_per_kwh, dtype: float64

## Feature / target builder

For a given `lead` (hours), build one row per target hour `h` where:
- `as_of = h - lead` (the simulated issue time)
- features are calendar facts about `h` (known perfectly at any horizon, docs/FORECASTING.md §4) plus price lags computed **relative to `as_of`**
- target is `price[h]`

A row is dropped if any required lag falls outside the data we have.

In [2]:
def calendar_features(target_times: pd.DatetimeIndex) -> pd.DataFrame:
    hour_of_day = target_times.hour + target_times.minute / 60.0
    return pd.DataFrame(
        {
            "hour_sin": np.sin(2 * np.pi * hour_of_day / 24),
            "hour_cos": np.cos(2 * np.pi * hour_of_day / 24),
            "day_of_week": target_times.dayofweek,
            "is_weekend": (target_times.dayofweek >= 5).astype(int),
        },
        index=target_times,
    )


def build_dataset(price: pd.Series, lead: int) -> pd.DataFrame:
    target_times = price.index
    as_of = target_times - pd.Timedelta(hours=lead)

    lag_24 = price.reindex(as_of - pd.Timedelta(hours=24)).to_numpy()
    lag_168 = price.reindex(as_of - pd.Timedelta(hours=168)).to_numpy()
    b1 = price.reindex(target_times - pd.Timedelta(hours=168)).to_numpy()  # seasonal naive, docs/FORECASTING.md §2

    feats = calendar_features(target_times)
    feats["lag_24"] = lag_24
    feats["lag_168"] = lag_168
    feats["b1_pred"] = b1
    feats["target"] = price.to_numpy()

    return feats.dropna()


FEATURE_COLS = ["hour_sin", "hour_cos", "day_of_week", "is_weekend", "lag_24", "lag_168"]
TEST_WEEKS = 8

In [3]:
def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


results = {}

for lead in (24, 48):
    data = build_dataset(price, lead)
    cutoff = data.index.max() - pd.Timedelta(weeks=TEST_WEEKS)
    train = data[data.index <= cutoff]
    test = data[data.index > cutoff]

    b1_mae = mae(test["target"].to_numpy(), test["b1_pred"].to_numpy())

    model = lgb.LGBMRegressor(
        objective="regression_l1",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=15,
        min_child_samples=20,
        verbosity=-1,
    )
    model.fit(train[FEATURE_COLS], train["target"])
    lgb_pred = model.predict(test[FEATURE_COLS])
    lgb_mae = mae(test["target"].to_numpy(), lgb_pred)

    skill = 1 - lgb_mae / b1_mae

    results[lead] = {
        "n_train": len(train),
        "n_test": len(test),
        "b1_mae_ore": b1_mae,
        "lgbm_mae_ore": lgb_mae,
        "skill_vs_b1": skill,
    }

    print(f"lead={lead}h  train={len(train)} test={len(test)}")
    print(f"  B1 (seasonal naive) MAE:  {b1_mae:.2f} ore/kWh (ex-VAT)")
    print(f"  LightGBM MAE:             {lgb_mae:.2f} ore/kWh (ex-VAT)")
    print(f"  skill = 1 - MAE_model/MAE_B1: {skill:+.3f}")
    print()

lead=24h  train=7224 test=1344
  B1 (seasonal naive) MAE:  25.96 ore/kWh (ex-VAT)
  LightGBM MAE:             28.38 ore/kWh (ex-VAT)
  skill = 1 - MAE_model/MAE_B1: -0.093



lead=48h  train=7200 test=1344
  B1 (seasonal naive) MAE:  25.96 ore/kWh (ex-VAT)
  LightGBM MAE:             28.76 ore/kWh (ex-VAT)
  skill = 1 - MAE_model/MAE_B1: -0.108



In [4]:
pd.DataFrame(results).T

,n_train,n_test,b1_mae_ore,lgbm_mae_ore,skill_vs_b1
24,7224.0,1344.0,25.958103,28.378835,-0.093255
48,7200.0,1344.0,25.958103,28.764553,-0.108115
